In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from tqdm import tqdm


# ----------------------------
# CONFIG
# ----------------------------
def find_project_root(start: Path = Path.cwd()):
    cur = start.resolve()
    # walk up a few levels to find a repo root (heuristics)
    for _ in range(10):
        if (cur / "data").exists() and (cur / "data" / "raw").exists():
            return cur
        if (cur / "requirements.txt").exists() or (cur / ".git").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()


PROJECT_ROOT = find_project_root()
print("Using PROJECT_ROOT:", PROJECT_ROOT)

ANNOTATIONS = PROJECT_ROOT / "data" / "raw" / "annotations.csv"

SOURCE_REPOS = {
    "37voy/Luna16s0456": ["subset0", "subset4", "subset5", "subset6"],
    "37voy/Luna16s1278": ["subset1", "subset2", "subset7", "subset78"],
    "37voy/Luna16s39": [
        "subset3",
        "subset9",
    ],
}

PROCESSED_REPO_ID = "37voy/Luna16processed"

print("ANNOTATIONS:", ANNOTATIONS, "exists:", ANNOTATIONS.exists())
print("SOURCE_REPOS:", SOURCE_REPOS)
print("PROCESSED_REPO_ID:", PROCESSED_REPO_ID)

# ----------------------------
# LOAD ANNOTATIONS
# ----------------------------
df = pd.read_csv(str(ANNOTATIONS))

# LUNA16 uses seriesuid to match scans
df.head()

Using PROJECT_ROOT: C:\Users\shiel\Documents\Python\LungModules
DATA_DIR: C:\Users\shiel\Documents\Python\LungModules\data\raw\subset0\subset0 exists: True
ANNOTATIONS: C:\Users\shiel\Documents\Python\LungModules\data\raw\annotations.csv exists: True
OUTPUT_DIR: C:\Users\shiel\Documents\Python\LungModules\data\processed\subset0 exists: True


,seriesuid,coordX,coordY,coordZ,diameter_mm
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-128.699421,-175.319272,-298.387506,5.651471
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,103.783651,-211.925149,-227.121250,4.224708
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,69.639017,-140.944586,876.374496,5.786348
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,-24.013824,192.102405,-391.081276,8.143262
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,2.441547,172.464881,-405.493732,18.545150


In [6]:
# load an mdh scan
def load_mhd(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)  # (z, y, x)
    spacing = img.GetSpacing()  # (x, y, z)
    origin = img.GetOrigin()
    direction = img.GetDirection()
    return arr, spacing, origin, direction

In [7]:
def normalize_hu(arr):
    arr = np.clip(arr, -1000, 400)
    arr = (arr + 1000) / 1400
    return arr.astype(np.float32)

In [8]:
def make_nodule_mask(scan_uid, arr, spacing, origin):
    mask = np.zeros_like(arr, dtype=np.uint8)

    nods = df[df["seriesuid"] == scan_uid]

    for _, row in nods.iterrows():
        world = np.array([row["coordZ"], row["coordY"], row["coordX"]])
        diameter = row["diameter_mm"]

        # convert world → voxel
        voxel = np.abs((world - origin[::-1]) / spacing[::-1]).astype(int)

        radius = int(diameter / 2 / spacing[0])

        z, y, x = voxel
        zz, yy, xx = np.ogrid[: arr.shape[0], : arr.shape[1], : arr.shape[2]]

        dist = (zz - z) ** 2 + (yy - y) ** 2 + (xx - x) ** 2
        mask[dist <= radius**2] = 1

    return mask

In [ ]:
from io import BytesIO

from huggingface_hub import HfApi, hf_hub_download


def sample_to_npz_bytes(img, mask, spacing, origin):
    buffer = BytesIO()
    np.savez_compressed(
        buffer,
        image=img,
        mask=mask,
        spacing=np.asarray(spacing, dtype=np.float32),
        origin=np.asarray(origin, dtype=np.float32),
    )
    buffer.seek(0)
    return buffer.getvalue()


def get_source_scan_path(repo_id, mhd_path):
    mhd_local = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=mhd_path)
    raw_path = mhd_path.replace(".mhd", ".raw")
    hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=raw_path)
    return mhd_local


def collect_source_scans(source_repos):
    """
    scan collector:
     - Handles nested folders
     - Ensures subset detection is correct
    """
    api = HfApi()
    scan_items = []

    # Flatten allowed subsets for quick lookup
    all_allowed = {subset for subsets in source_repos.values() for subset in subsets}

    print("Allowed subsets:", all_allowed)

    for repo_id, subset_names in source_repos.items():
        print(f"\nScanning repo: {repo_id}")
        files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")

        for file_name in files:
            if not file_name.endswith(".mhd"):
                continue

            parts = Path(file_name).parts

            # Find the FIRST folder in the path that matches a subset name
            subset_name = None
            for p in parts:
                if p in all_allowed:
                    subset_name = p
                    break

            if subset_name is None:
                print(f"⚠️  Skipping file with no valid subset: {file_name}")
                continue

            # Extract scan UID
            scan_uid = Path(file_name).stem

            scan_items.append((repo_id, subset_name, file_name))

            print(f"✓ Found scan: subset={subset_name}, uid={scan_uid}")

    print(f"\nTotal scans found: {len(scan_items)}")
    return scan_items


def upload_processed_subset(scan_items, repo_id, subdir="processed"):
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

    preview_npz_bytes = None

    for source_repo_id, subset_name, file_name in tqdm(scan_items):
        scan_uid = Path(file_name).stem
        path = get_source_scan_path(source_repo_id, file_name)

        arr, spacing, origin, _direction = load_mhd(path)
        arr_norm = normalize_hu(arr)
        mask = make_nodule_mask(scan_uid, arr, spacing, origin)

        npz_bytes = sample_to_npz_bytes(arr_norm, mask, spacing, origin)
        if preview_npz_bytes is None:
            preview_npz_bytes = npz_bytes

        api.upload_file(
            path_or_fileobj=BytesIO(npz_bytes),
            path_in_repo=f"{subdir}/{subset_name}/{scan_uid}.npz",
            repo_id=repo_id,
            repo_type="dataset",
            commit_message=f"Add processed scan {scan_uid}",
        )

    return preview_npz_bytes


scan_items = collect_source_scans(SOURCE_REPOS)
preview_npz_bytes = upload_processed_subset(scan_items, PROCESSED_REPO_ID)
print(f"Uploaded {len(scan_items)} scans to {PROCESSED_REPO_ID}")

  8%|▊         | 7/89 [02:56<34:32, 25.27s/it]


OSError: [Errno 28] No space left on device

In [ ]:
sample = np.load(BytesIO(preview_npz_bytes))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(sample["image"][50], cmap="gray")
plt.title("CT Slice")

plt.subplot(1, 2, 2)
plt.imshow(sample["mask"][50], cmap="gray")
plt.title("Nodule Mask")
plt.show()